# Import Libaries and Functions

In [1]:
from _utils import *

# XGB model 
- Link to 2025-03-07 results: https://docs.google.com/presentation/d/1glchNMC5hmDNKWJJXEZWrfv2TXf5l8j0cvdH7K_p4xY/edit?slide=id.g2e3c020b9c3_0_256#slide=id.g2e3c020b9c3_0_256

## Splitting into Train and Test Sets

In [2]:
set_seed(SEED)

# Load preprocessed data
df_pre = (
    pd.read_csv("../data/pipeline/preprocessed.csv")
      .query("file_hash not in @EXTERNAL_VALIDATION_SET")
      .query("new_subtype_label not in @RARE_SUBTYPES")
      .drop(columns=["rare_subtype"])
      .set_index("file_hash")
)


# Get the correct type labels from ant_df_long
type_lookup = ant_df_long[['file_hash', 'original_type_label']].drop_duplicates()
df_pre_with_type = df_pre.reset_index().merge(type_lookup, on='file_hash', how='left').set_index('file_hash')

# Use original_type_label instead of regex extraction
df_pre_with_type["type"] = df_pre_with_type["original_type_label"].astype(str)
df_pre_with_type["subtype"] = df_pre_with_type["new_subtype_label"].astype(str)

# Split features and labels
X = df_pre_with_type.drop(columns=["new_subtype_label", "original_type_label", "type", "subtype"])
y_type = df_pre_with_type["type"].astype(str)
y_subtype = df_pre_with_type["subtype"].astype(str)


# Train/test split (stratify by subtype like before)
X_train, X_test, y_type_train, y_type_test, y_subtype_train, y_subtype_test = train_test_split(
    X, y_type, y_subtype, test_size=0.20, stratify=y_subtype, random_state=SEED
)

# Clean column names + enforce numeric
X_train = clean_numeric_columns(X_train)
X_test = X_test.copy()
X_test.columns = X_train.columns


UndefinedVariableError: local variable 'EXTERNAL_VALIDATION_SET' is not defined

In [ ]:
# Build crosswalk for training set
cw_train = (
    pd.DataFrame({
        "file_hash": X_train.index,
        "type": y_type_train.values,
        "subtype": y_subtype_train.values,
        "split": "train"
    })
)

# Build crosswalk for test set
cw_test = (
    pd.DataFrame({
        "file_hash": X_test.index,
        "type": y_type_test.values,
        "subtype": y_subtype_test.values,
        "split": "test"
    })
)

# Concatenate into one crosswalk
cw = pd.concat([cw_train, cw_test])

# Save to CSV
cw.to_csv("../data/pipeline/crosswalk.csv", index=False)


## Feature Selection Pipeline (Feature-Engine)

In [ ]:
# Encode subtypes for selection scoring
subtype_encoder, y_subtype_train_enc, y_subtype_test_enc = encode_labels(y_subtype_train, y_subtype_test)

# Drop columns with ≥99% missing values
threshold = 0.99
missing_fraction = X_train.isna().mean()
cols_to_drop = missing_fraction[missing_fraction >= threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns with ≥{int(threshold*100)}% missingness.")

X_train_clean = X_train.drop(columns=cols_to_drop)
X_test_clean  = X_test.drop(columns=cols_to_drop, errors="ignore")

# Lists AFTER dropping
binary_list     = [c for c in X_train_clean.columns if c.endswith("_yn")]
continuous_list = [c for c in X_train_clean.columns if any(s in c for s in ["_min", "_max", "_simfeat"])]
discrete_list   = [c for c in X_train_clean.columns if c.endswith("_count")]

# Cast binaries to category
X_train_clean[binary_list] = X_train_clean[binary_list].astype("category")
X_test_clean[binary_list]  = X_test_clean[binary_list].astype("category")

# Preprocess pipeline (impute/scale/discretize)
feature_preprocess_pipeline = Pipeline(steps=[
    ("missing_indicator", AddMissingIndicator(variables=continuous_list)),
    ("arbitrary_number_imputer", ArbitraryNumberImputer(arbitrary_number=0, variables=discrete_list)),
    ("median_imputer", MeanMedianImputer(imputation_method="median", variables=continuous_list)),
    ("binary_imputer", CategoricalImputer(fill_value="0", variables=binary_list)),
    ("normalize", SklearnTransformerWrapper(MinMaxScaler(feature_range=(0, 1)), variables=continuous_list)),
    ("discretize", DecisionTreeDiscretiser(cv=5, scoring="roc_auc_ovr", regression=False, random_state=SEED, variables=discrete_list)),
])

X_train_pre = feature_preprocess_pipeline.fit_transform(X_train_clean, y_subtype_train_enc)
X_test_pre  = feature_preprocess_pipeline.transform(X_test_clean)

# Feature selection pipeline (same as yours)
feature_selection_pipeline = Pipeline(steps=[
    ("drop_constant", DropConstantFeatures(tol=0.99)),
    ("drop_duplicates", DropDuplicateFeatures()),
    ("correlated_features", SmartCorrelatedSelection(
        method="pearson",
        threshold=0.9,
        selection_method="model_performance",
        estimator=xgb.XGBClassifier(random_state=0),
        scoring="roc_auc_ovr"
    )),
])

X_train_selected = feature_selection_pipeline.fit_transform(X_train_pre, y_subtype_train_enc)
X_test_selected  = feature_selection_pipeline.transform(X_test_pre)

# Convert any remaining categories to ints (XGBoost safety)
for df in (X_train_selected, X_test_selected):
    cat_cols = df.select_dtypes(include="category").columns
    if len(cat_cols):
        df[cat_cols] = df[cat_cols].astype(int)

# Final matrices used by both approaches
X_train_final = X_train_selected.copy()
X_test_final  = X_test_selected.copy()


a = set(X_test)
b = set(X_test_selected)
DROPPED_FEATURES = list(a-b)
print("Dropped columns", len(DROPPED_FEATURES))

In [ ]:
len(X_test.columns)

In [ ]:
# Features as rows
feature_dd = pd.DataFrame(df_pre.columns, columns=["feature"]).reset_index(drop=True)
feature_dd["dropped"] = feature_dd["feature"].isin(DROPPED_FEATURES)
feature_dd.to_csv("../data/pipeline/feature_dd.csv", index=False)

## Bottom-Up Reverse Hierchical 

In [ ]:
# --- CV splitter (yours) ---
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)

# --- Base estimator ---
base_xgb = xgb.XGBClassifier(
    objective="multi:softmax",
    num_class=len(subtype_encoder.classes_),
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=1,
    # optional: faster histogram algorithm if available
    tree_method="hist"
)

# --- Hyperparameter search space ---
# Keep ranges modest to balance exploration vs. runtime.
param_distributions = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.1, 0.2],
    "n_estimators": [200, 400, 800, 1200],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.5, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.5, 1.0],
    "reg_alpha": [0.0, 0.01, 0.1, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0],
}

# --- RandomizedSearchCV (uses your cv) ---
from sklearn.model_selection import RandomizedSearchCV
search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=40,                 # increase/decrease as needed
    scoring="accuracy",
    cv=cv,
    random_state=SEED,
    n_jobs=1,                  # keep 1 if you need strict reproducibility
    refit=True,                # refit best params on full training set
    verbose=1
)

# --- Run search ---
search.fit(X_train_final, y_subtype_train_enc)
print(f"[Reverse] Best CV accuracy: {search.best_score_:.4f}")
print("[Reverse] Best params:", search.best_params_)

# Best tuned model already refit on full training data
subtype_model = search.best_estimator_

# --- Predict subtype ---
y_subtype_pred_enc_rev = subtype_model.predict(X_test_final)
y_subtype_pred_rev = subtype_encoder.inverse_transform(y_subtype_pred_enc_rev)

# --- Derive TYPE from predicted subtype using lookup ---
# Create a mapping from subtype to type using your ground truth
subtype_to_type_map = ant_df_long[['new_subtype_label', 'original_type_label']].drop_duplicates().set_index('new_subtype_label')['original_type_label'].to_dict()

# Map XGB's predicted subtypes to their correct types
y_type_pred_rev = pd.Series(y_subtype_pred_rev).map(subtype_to_type_map)

# --- Evaluation ---
print("\n=== Held-Out Evaluation (Reverse: subtype->type) ===")
print("Type Accuracy (derived from subtype):", accuracy_score(y_type_test, y_type_pred_rev))
print("\nSubtype Accuracy (overall):", accuracy_score(y_subtype_test, y_subtype_pred_enc_rev))
print("\nSubtype Classification Report (overall):")
print(classification_report(y_subtype_test, y_subtype_pred_rev))

# --- Confusion matrices ---
type_labels = sorted(pd.Series(y_type_test).unique())
plot_confusion(y_type_test, y_type_pred_rev, labels=type_labels,
               title="Derived TYPE Confusion Matrix (Counts)", figsize=(8, 6))
plot_confusion(y_type_test, y_type_pred_rev, labels=type_labels,
               title="Derived TYPE Confusion Matrix (Row-Normalized)", normalize="true", figsize=(8, 6))

subtype_labels = sorted(pd.Series(y_subtype_test).unique())
plot_confusion(y_subtype_test, y_subtype_pred_rev, labels=subtype_labels,
               title="SUBTYPE Confusion Matrix (Counts) — Reverse", figsize=(14, 10))
plot_confusion(y_subtype_test, y_subtype_pred_rev, labels=subtype_labels,
               title="SUBTYPE Confusion Matrix (Row-Normalized) — Reverse", normalize="true", figsize=(14, 10))

# --- Feature importance (reverse) ---
subtype_feat_imp_rev = xgb_gain_importance(subtype_model, X_train_final.columns)
plot_top_features(subtype_feat_imp_rev, "Top 15 Features")

In [ ]:
booster = subtype_model.get_booster()

gain  = booster.get_score(importance_type="gain")
cover = booster.get_score(importance_type="cover")
weight = booster.get_score(importance_type="weight")

fi_csv = pd.DataFrame({
    "feature": list(gain.keys()),
    "gain": list(gain.values()),
    "cover": [cover.get(f, 0) for f in gain.keys()],
    "weight": [weight.get(f, 0) for f in gain.keys()]
})

fi_csv.to_csv("../data/pipeline/feature_importance_global.csv", index=False)

In [ ]:
#fi_names = fi_csv[["feature"]]

In [ ]:
#sheet = client.open("label_df")

# --- Select or create Sheet2 ---
#try:
#    worksheet = sheet.worksheet("Sheet2")  # If Sheet2 already exists
#except gspread.exceptions.WorksheetNotFound:
#    worksheet = sheet.add_worksheet(title="Sheet2", rows="1000", cols="20")


# --- Clear existing data & write new data ---
#worksheet.clear()
#worksheet.update([fi_names.columns.values.tolist()] + fi_names.values.tolist())

## SHAP Feature Importance

In [ ]:
import numpy as np
import shap

# Build explainer (interventional is usually best for tree models)
explainer = shap.TreeExplainer(subtype_model, feature_perturbation="interventional")

# SHAP values for test set
shap_values = explainer.shap_values(X_test_final)  # list-of-arrays (classic) OR 3D array (newer SHAP)

# Predicted class positions per row (aligns with shap_values class order)
proba = subtype_model.predict_proba(X_test_final)          # shape (n_samples, n_classes)
pred_pos = np.asarray(proba).argmax(axis=1)                # shape (n_samples,)

n_samples, n_features = X_test_final.shape

# Get per-row SHAP for the predicted class, handling both SHAP formats
if isinstance(shap_values, list):
    # shap_values[k] -> (n_samples, n_features) for class k
    shap_pred = np.vstack([shap_values[k][i] for i, k in enumerate(pred_pos)])  # (n_samples, n_features)
else:
    # shap_values -> (n_samples, n_features, n_classes)
    shap_pred = shap_values[np.arange(n_samples), :, pred_pos]                  # (n_samples, n_features)

# Pack into a dataframe
df_shap = pd.DataFrame(shap_pred, columns=X_test_final.columns)
df_shap["hash"] = X_test_final.index


In [ ]:
# Create an explainer
#explainer = shap.Explainer(xgb_model, X_test)
# Compute SHAP values
#shap_values = explainer(X_test)
#plt.title("I H")
#plt.title("I K A-type")
#plt.title("I KDR")
#shap.plots.beeswarm(shap_values[:,:,5])

## Combining XGB and GPT predictions together

In [ ]:
# === Build DataFrame with predictions and truth ===

# True values from test set
true_subtype = y_subtype_test  # string labels
true_type = y_type_test        # string labels

# Build dataframe using reverse model predictions
df_xgb_pred = pd.DataFrame({
    "hash": X_test_selected.index,  # or X_test_final.index if X_test_selected isn't defined
    "true_type": true_type.values,
    "true_subtype": true_subtype.values,
    "xgb_pred_type": y_type_pred_rev.values,
    "xgb_pred_subtype": y_subtype_pred_rev
})

# Add match flags
df_xgb_pred["xgb_type_match"] = df_xgb_pred["true_type"] == df_xgb_pred["xgb_pred_type"]
df_xgb_pred["xgb_subtype_match"] = df_xgb_pred["true_subtype"] == df_xgb_pred["xgb_pred_subtype"]
df_both = df_xgb_pred.merge(gpt_df3, how="left", on="hash")
df_both["gpt_type_match"] = df_both["true_type"] == df_both["gpt_pred_type"]

df_both["gpt_subtype_match"] = df_both["true_subtype"] == df_both["gpt_pred_subtype"]


In [ ]:
df_xgb_pred

In [ ]:

# Optional: add confidence for convenience in R
max_prob = proba.max(axis=1)                       # model confidence of predicted class
df_conf = pd.DataFrame({"hash": X_test_final.index, "xgb_pred_prob": max_prob})

# Merge into df_both (which already has truth/preds/match flags)
df_both = df_both.merge(df_shap, on="hash", how="left").merge(df_conf, on="hash", how="left")

# Export for R
df_both.to_csv("../data/pipeline/predictions_with_shap.csv", index=False)


In [ ]:
df_both

In [ ]:
#Checking Misclassfied XGB Samples
df_wrong = df_xgb_pred[
    (df_xgb_pred["xgb_type_match"] == False) |
    (df_xgb_pred["xgb_subtype_match"] == False)
]
MIS_TYPES = (
    df_wrong
    .query("xgb_type_match == False")["hash"]
    .tolist()
)
MIS_SUBTYPES = (
    df_wrong
    .query("xgb_subtype_match == False")["hash"]
    .tolist()
)

#df_wrong


# GPT performance

In [ ]:
df_both.columns

In [ ]:
#df_both[df_both["gpt_pred_subtype"].isna()]

In [ ]:
# --- helper: reusable confusion matrix plotter ---
def plot_confusion(y_true, y_pred, title, normalize=None, figsize=(10, 7)):
    # Collect labels present in either true or predicted
    labels = sorted(pd.unique(pd.concat([pd.Series(y_true), pd.Series(y_pred)])))
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize=normalize)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)

    plt.figure(figsize=figsize)
    fmt = ".2f" if normalize else "d"
    sns.heatmap(cm_df, annot=True, fmt=fmt, cmap="Blues", linewidths=0.5)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# === Overall GPT Type Accuracy ===
gpt_type_acc = accuracy_score(df_both["true_type"], df_both["gpt_pred_type"])
print(f"GPT Type Accuracy: {gpt_type_acc:.4f}")

print("\nGPT Type Classification Report:")
print(classification_report(
    df_both["true_type"],
    df_both["gpt_pred_type"]
))

# --- Type confusion matrices (counts + row-normalized) ---
plot_confusion(
    df_both["true_type"],
    df_both["gpt_pred_type"],
    title="GPT Type Confusion Matrix (Counts)",
    normalize=None
)

plot_confusion(
    df_both["true_type"],
    df_both["gpt_pred_type"],
    title="GPT Type Confusion Matrix (Row-Normalized)",
    normalize="true"
)

# === Overall GPT Subtype Accuracy (no conditioning) ===
gpt_subtype_acc = accuracy_score(
    df_both["true_subtype"], df_both["gpt_pred_subtype"]
)
print(f"\nGPT Subtype Accuracy (Overall): {gpt_subtype_acc:.4f}")

print("\nGPT Subtype Classification Report (Overall):")
print(classification_report(
    df_both["true_subtype"],
    df_both["gpt_pred_subtype"]
))

# --- Subtype confusion matrices (overall) ---
plot_confusion(
    df_both["true_subtype"],
    df_both["gpt_pred_subtype"],
    title="GPT Subtype Confusion Matrix (Counts)",
    normalize=None
)

plot_confusion(
    df_both["true_subtype"],
    df_both["gpt_pred_subtype"],
    title="GPT Subtype Confusion Matrix (Row-Normalized)",
    normalize="true"
)


# Merging them

In [ ]:
# ====== helpers ======
def _labels_from(*series_list):
    s = pd.concat([pd.Series(s) for s in series_list], ignore_index=True)
    return sorted(s.dropna().unique().tolist())

def plot_cm(ax, y_true, y_pred, labels, title, normalize=None, annot=True):
    if len(y_true) == 0:
        ax.axis('off')
        ax.text(0.5, 0.5, "No samples", ha="center", va="center", fontsize=12)
        ax.set_title(title)
        return
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize=normalize)
    fmt = ".2f" if normalize else "d"
    sns.heatmap(cm, ax=ax, annot=annot, fmt=fmt, cmap="Blues",
                xticklabels=labels, yticklabels=labels, linewidths=0.5)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis='x', rotation=90)

# ====== assemble labels ======
type_labels = _labels_from(
    df_both["true_type"], 
    df_both["gpt_pred_type"], 
    df_both["xgb_pred_type"]
)
subtype_labels = _labels_from(
    df_both["true_subtype"], 
    df_both["gpt_pred_subtype"], 
    df_both["xgb_pred_subtype"]
)

# ====== accuracies ======
acc_type_gpt = accuracy_score(df_both["true_type"], df_both["gpt_pred_type"])
acc_type_xgb = accuracy_score(df_both["true_type"], df_both["xgb_pred_type"])

acc_sub_gpt = accuracy_score(df_both["true_subtype"], df_both["gpt_pred_subtype"])
acc_sub_xgb = accuracy_score(df_both["true_subtype"], df_both["xgb_pred_subtype"])

# ====== 2x2 confusion matrices: Type (XGB/GPT) + Subtype (XGB/GPT) ======
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Type — XGB left, GPT right
plot_cm(
    axes[0, 0],
    df_both["true_type"], df_both["xgb_pred_type"], type_labels,
    title=f"Type Confusion (XGB) — Acc={acc_type_xgb:.3f}"
)
plot_cm(
    axes[0, 1],
    df_both["true_type"], df_both["gpt_pred_type"], type_labels,
    title=f"Type Confusion (GPT) — Acc={acc_type_gpt:.3f}"
)

# Subtype — XGB left, GPT right (no conditioning)
plot_cm(
    axes[1, 0],
    df_both["true_subtype"], df_both["xgb_pred_subtype"], subtype_labels,
    title=f"Subtype Confusion (XGB) — Acc={acc_sub_xgb:.3f}"
)
plot_cm(
    axes[1, 1],
    df_both["true_subtype"], df_both["gpt_pred_subtype"], subtype_labels,
    title=f"Subtype Confusion (GPT) — Acc={acc_sub_gpt:.3f}"
)

fig.suptitle("XGB vs GPT — Type and Subtype Confusion Matrices (Overall)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# ====== Summary accuracy bar chart ======
acc_df = pd.DataFrame({
    "Model": ["XGB", "GPT", "XGB", "GPT"],
    "Level": ["Type", "Type", "Subtype", "Subtype"],
    "Accuracy": [acc_type_xgb, acc_type_gpt, acc_sub_xgb, acc_sub_gpt]
})

plt.figure(figsize=(8, 5))
sns.barplot(data=acc_df, x="Level", y="Accuracy", hue="Model", hue_order=["XGB", "GPT"])
plt.ylim(0, 1)
plt.title("Accuracy Summary: Type vs Subtype (Overall)")
for i, v in enumerate(acc_df["Accuracy"]):
    plt.text(x=i//2 + (-0.15 if i % 2 == 0 else 0.15),
             y=min(0.98, (0 if np.isnan(v) else v)) + 0.01,
             s=("NA" if np.isnan(v) else f"{v:.2f}"),
             ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
#76.9,74.4

In [ ]:
!git add .
!git commit -m "streaminglining code"
!git push 